In [95]:
from langgraph.graph import StateGraph, END, START
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

In [96]:
llm = ChatOllama(model = "llama3.2:1b")

In [97]:
class JokeState(TypedDict):
    title: str
    joke: str
    explanation: str

In [98]:
def get_joke(state: JokeState):
    prompt = f"Give me one funny joke about this topic: {state['title']}"
    res = llm.invoke(prompt)

    print("get joke done")

    return {"joke" : res.content}

In [99]:
def get_explanation(state: JokeState):
    prompt = f"Give me the explanation of below joke: \n {state['joke']}"
    res = llm.invoke(prompt)
    time.sleep(5)
    print("get_explanation done")

    return {"explanation" : res.content}

In [100]:
graph = StateGraph(JokeState)

graph.add_node("get_joke", get_joke)
graph.add_node("get_explanation", get_explanation)

graph.add_edge(START, "get_joke")
graph.add_edge("get_joke", "get_explanation")
graph.add_edge("get_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [101]:
config1 = {"configurable": {"thread_id" : "5"}}

workflow.invoke({"title": "Pizza"},  config=config1)

get joke done
get_explanation done


{'title': 'Pizza',
 'joke': 'Why did the pizza go to therapy? Because it was feeling crusty under the pressure.',
 'explanation': 'This joke is a play on words. "Crusty" has a double meaning here. In baking, crust is a part of a pizza that is crispy on the outside and soft on the inside. But in this joke, "crusty" also sounds like "crusty under pressure," which is a phrase that means feeling stressed or overwhelmed.\n\nSo, the joke is saying that the pizza is feeling stressed or overwhelmed because it\'s feeling like it\'s under too much pressure, which is a common way to describe feeling "crusty" in a baking sense. It\'s a clever play on words that uses a bit of wordplay to create a pun.'}

In [120]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'title': 'Pizza'}, next=('get_joke',), config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76fb-64df-6548-8002-3e4af350d5ce'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-11-03T10:17:02.509999+00:00', parent_config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76ea-6712-6f32-8001-5bd2964ce20b'}}, tasks=(PregelTask(id='ba903b91-b665-2795-8543-e0566959fff2', name='get_joke', path=('__pregel_pull', 'get_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'title': 'Pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning here. A pizza is typically considered to be a bit "crusty" because it\'s crispy on the outside and chewy on the inside. However, in this joke, "crusty" is also used as an idiomatic expressi

In [103]:
workflow.invoke(None, config=config1) #fault tolerence

{'title': 'Pizza',
 'joke': 'Why did the pizza go to therapy? Because it was feeling crusty under the pressure.',
 'explanation': 'This joke is a play on words. "Crusty" has a double meaning here. In baking, crust is a part of a pizza that is crispy on the outside and soft on the inside. But in this joke, "crusty" also sounds like "crusty under pressure," which is a phrase that means feeling stressed or overwhelmed.\n\nSo, the joke is saying that the pizza is feeling stressed or overwhelmed because it\'s feeling like it\'s under too much pressure, which is a common way to describe feeling "crusty" in a baking sense. It\'s a clever play on words that uses a bit of wordplay to create a pun.'}

In [109]:
state = workflow.get_state(config1)

In [125]:
workflow.update_state(
    {"configurable": {"thread_id" : "5", 'checkpoint_ns': '', 'checkpoint_id': '1f1d76ea-6712-6f32-8001-5bd2964ce20b'}},
    {"title": "samosa"}
)

{'configurable': {'thread_id': '5',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d76fe-3c21-65d8-8002-ff03267b4b10'}}

In [126]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'title': 'samosa'}, next=('get_joke',), config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76fe-3c21-65d8-8002-ff03267b4b10'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-11-03T10:18:18.768520+00:00', parent_config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76ea-6712-6f32-8001-5bd2964ce20b'}}, tasks=(PregelTask(id='4c4390de-c521-ae9d-9dac-664928a969c8', name='get_joke', path=('__pregel_pull', 'get_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'title': 'Pizza'}, next=('get_joke',), config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76fd-2491-6210-8002-536ffe32ceb2'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-11-03T10:17:49.454172+00:00', parent_config={'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f1d76ea-6712-6f3

In [127]:
workflow.invoke(None, {**config1, "configurable": {**config1["configurable"], "checkpoint_id" : "1f1d76fe-3c21-65d8-8002-ff03267b4b10"}})

get joke done
get_explanation done


{'title': 'samosa',
 'joke': 'Here\'s one: Why did the samosa go to therapy?\n\nBecause it was feeling a little "spiced" up and was struggling to "fill" in the gaps of its life!',
 'explanation': 'This joke is a play on words. A samosa is a type of Indian snack food that is typically wrapped in a dough pocket and filled with spiced potatoes, peas, and other ingredients. The phrase "fill in the gaps" is an idiom that means to try to complete or make up for something that is lacking.\n\nIn this joke, the samosa is going to therapy because it\'s feeling a little "spiced" up, which is a pun on the fact that samosas are typically spiced food. It\'s also using the word "fill" in a literal sense, as the samosa is a folded food item that needs to be "filled" with fillings. The joke is humorous because it takes a common phrase and gives it a silly, unexpected twist.'}